In [0]:

file_path = "/Volumes/wsaditya/default/vol2" 
df_csv = spark.read.csv(file_path, header=True, inferSchema=True)


cleaned_columns = [col.replace(' ', '_').replace('-', '_') for col in df_csv.columns]
df_csv = df_csv.toDF(*cleaned_columns)


target_table_name = "default.superstore_target"
(df_csv.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")  
    .saveAsTable(target_table_name)
)

print("Target table successfully overwritten with new metadata!")

Target table successfully overwritten with new metadata!


In [0]:
dummy_records = [
    (1, "CA-2016-152156", "Claire Gute Updated", 500.00), 
    

    (2, "CA-2016-152156", "Claire Updated Again", 800.00),
    (3, "CA-2016-138688", "Darrin Van Huff", 99.99),
    

    (4, "US-2015-108966", "Sean O'Donnell", 957.5775),
    (5, "US-2015-108966", "Sean O'Donnell", 22.368),
    

    (6, "CA-2014-115812", "Brosina Hoffman", None),
    (7, "CA-2014-115812", None, 7.28),
    

    (999999, "CA-2026-TEST", "Insert Tester", 250.00),
    (1000001, "NEW-2026-001", "Alice Smith", 120.50),
    (1000002, "NEW-2026-001", "Alice Smith", 45.00),
    (1000003, "NEW-2026-002", "Bob Jones", 999.99),
    (1000004, "NEW-2026-003", "Charlie Brown", 10.00),
    (1000005, "NEW-2026-004", "Diana Prince", 550.75),
    (1000006, "NEW-2026-004", "Diana Prince", 15.25),
    

    (1000007, "NEW-2026-005", None, None)
]

columns = ["Row_ID", "Order_ID", "Customer_Name", "Sales"]

df_dummy_source = spark.createDataFrame(dummy_records, columns)
display(df_dummy_source)

Row_ID,Order_ID,Customer_Name,Sales
1,CA-2016-152156,Claire Gute Updated,500.0
2,CA-2016-152156,Claire Updated Again,800.0
3,CA-2016-138688,Darrin Van Huff,99.99
4,US-2015-108966,Sean O'Donnell,957.5775
5,US-2015-108966,Sean O'Donnell,22.368
6,CA-2014-115812,Brosina Hoffman,null
7,CA-2014-115812,null,7.28
999999,CA-2026-TEST,Insert Tester,250.0
1000001,NEW-2026-001,Alice Smith,120.5
1000002,NEW-2026-001,Alice Smith,45.0


In [0]:
from delta.tables import *


delta_target = DeltaTable.forName(spark, "default.superstore_target")


(delta_target.alias("target")
  .merge(
    df_dummy_source.alias("source"),
    "target.Row_ID = source.Row_ID"
  )
  .whenMatchedUpdate(set = {
    "Order_ID": "source.Order_ID",
    "Customer_Name": "source.Customer_Name",
    "Sales": "source.Sales"
  })
  .whenNotMatchedInsert(values = {
    "Row_ID": "source.Row_ID",
    "Order_ID": "source.Order_ID",
    "Customer_Name": "source.Customer_Name",
    "Sales": "source.Sales"
  })
  .execute()
)

print("15-Row Merge successfully completed!")


15-Row Merge successfully completed!


In [0]:
display(spark.sql("""
    SELECT Row_ID, Order_ID, Customer_Name, Sales 
    FROM default.superstore_target 
    WHERE Row_ID <= 7 OR Row_ID >= 999999
    ORDER BY Row_ID
"""))


Row_ID,Order_ID,Customer_Name,Sales
1,CA-2016-152156,Claire Gute Updated,500.0
2,CA-2016-152156,Claire Updated Again,800.0
3,CA-2016-138688,Darrin Van Huff,99.99
4,US-2015-108966,Sean O'Donnell,957.5775
5,US-2015-108966,Sean O'Donnell,22.368
6,CA-2014-115812,Brosina Hoffman,null
7,CA-2014-115812,null,7.28
999999,CA-2026-TEST,Insert Tester,250.0
1000001,NEW-2026-001,Alice Smith,120.5
1000002,NEW-2026-001,Alice Smith,45.0
